In [ ]:
import os
import time
import glob
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

start_time = time.time()
tf.keras.utils.set_random_seed(42)

# ---------------------------------------------------------
# 1. REUSE EXTRACTED DATA OR EXTRACT ONCE & LOAD CSV
# ---------------------------------------------------------
extract_dir = '/content/gtzan_data'
zip_matches = glob.glob('/content/**/*.zip', recursive=True)

if not os.path.exists(extract_dir) and len(zip_matches) > 0:
    print(f"Extracting {zip_matches[0]}...")
    with zipfile.ZipFile(zip_matches[0], 'r') as z:
        z.extractall(extract_dir)

csv_files = [
    p for p in glob.glob('/content/**/*.csv', recursive=True)
    if 'sample_data' not in p
]
assert len(csv_files) > 0, "No GTZAN CSV found. Upload the GTZAN zip or CSV to Colab Files."

csv_files.sort(key=lambda p: (0 if '3_sec' in os.path.basename(p) else 1, os.path.getsize(p)), reverse=True)
csv_path = csv_files[0]
print("Loaded dataset CSV:", csv_path)

df = pd.read_csv(csv_path)
label_col = 'label' if 'label' in df.columns else df.columns[-1]
drop_cols = [c for c in ['filename', 'length', label_col] if c in df.columns]

X_raw = df.drop(columns=drop_cols).select_dtypes(include=[np.number]).fillna(0.0).values.astype(np.float32)
y_raw = df[label_col].astype(str).values

encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)
class_names = list(encoder.classes_)
print(f"Samples: {len(X_raw)} | Features: {X_raw.shape[1]} | Genres: {len(class_names)}")

# ---------------------------------------------------------
# 2. STRATIFIED SPLIT, STANDARDIZE & SHAPE FOR LSTM
# ---------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

timesteps = 6
n_features = X_train_scaled.shape[1]
step_dim = int(np.ceil(n_features / timesteps))
pad_len = timesteps * step_dim - n_features

if pad_len > 0:
    X_train_scaled = np.pad(X_train_scaled, ((0, 0), (0, pad_len)), mode='constant')
    X_val_scaled = np.pad(X_val_scaled, ((0, 0), (0, pad_len)), mode='constant')

X_train_lstm = X_train_scaled.reshape(-1, timesteps, step_dim)
X_val_lstm = X_val_scaled.reshape(-1, timesteps, step_dim)

# ---------------------------------------------------------
# 3. DEDICATED STACKED LSTM MODEL ON CPU
# ---------------------------------------------------------
lstm_model = models.Sequential([
    layers.Input(shape=(timesteps, step_dim)),
    layers.LSTM(64, return_sequences=True, dropout=0.2),
    layers.LSTM(32, dropout=0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax')
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_accuracy', patience=4, restore_best_weights=True
)

history = lstm_model.fit(
    X_train_lstm, y_train,
    validation_data=(X_val_lstm, y_val),
    epochs=18,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

_, val_acc = lstm_model.evaluate(X_val_lstm, y_val, verbose=0)
elapsed = time.time() - start_time
print(f"\nLSTM Validation Accuracy: {val_acc * 100:.2f}%")
print(f"Total Execution Time: {elapsed:.1f} seconds")

# ---------------------------------------------------------
# 4. ACCURACY, LOSS & CONFUSION MATRIX
# ---------------------------------------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('GTZAN LSTM Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('GTZAN LSTM Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

val_probs = lstm_model.predict(X_val_lstm, verbose=0)
val_preds = np.argmax(val_probs, axis=1)

cm = confusion_matrix(y_val, val_preds)
fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.title('GTZAN LSTM Confusion Matrix')
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 5. SAMPLE PREDICTIONS (P = Predicted, T = True)
# ---------------------------------------------------------
print("\nSample Predictions (P = Predicted, T = True):")
for i in range(10):
    p_name = class_names[val_preds[i]]
    t_name = class_names[y_val[i]]
    conf = val_probs[i][val_preds[i]] * 100.0
    status = "CORRECT" if val_preds[i] == y_val[i] else "WRONG"
    print(f"Sample {i+1:02d} -> P: {p_name:10s} ({conf:4.1f}%) | T: {t_name:10s} [{status}]")